# Initial Datahandeling

### This notebook shows the initial handling of utterances in the Danish Parliament.
### It includes the following:

- Loading the two overlapping datasets of utterences in the Danish Parliament
    - Due to the ParlLawSpeech data being in .rds format, it was read into r, converted to .csv format and then saved in this repo
- Merging them by end data of corp_v2
- Sentence segmentation
- Reformatting to json object for easier handling during labeling for validation, training and inference
- Datacleaning
- Splitting into training data (including validationset to be extracted), and inference data

In [1]:
# Load packages

import pandas as pd
import os


# Corp_Folketing_V2

In [2]:
#read the corp folketing 

corp_v2_path = os.path.join("..",
                            "..",
                            "..",
                            "..",
                            "data",
                            "csv_meetings",
                            "Corp_Folketing_V2.csv")

# Read the two datasets
df_corp = pd.read_csv(corp_v2_path)
#x = df_corp.pop("Unnamed: 0") #dummy variable

#make date a datetime object
df_corp["date"] = pd.to_datetime(df_corp["date"])

In [3]:
#This dataset spans the following time period

min_time_corp = df_corp["date"].min()
max_time_corp = df_corp["date"].max()

print(f"The Corp dataset has coverage from {min_time_corp.date()} to {max_time_corp.date()}")

The Corp dataset has coverage from 1997-10-07 to 2018-12-20


# XML speeches

In [18]:
#read the PLS data. 
# OBS has been made to .csv in file ./PLS_rds_to_csv.RMD

xml_path = os.path.join("..",
                            "..",
                            "..",
                            "..",
                            "data",
                            "csv_meetings",
                            "meetings.csv")

# Read the two datasets
df_xml = pd.read_csv(xml_path)

#make date a datetime object
df_xml["date"] = pd.to_datetime(df_xml["date"])

In [19]:
#This dataset spans the following time period

min_time_xml = df_xml["date"].min()
max_time_xml = df_xml["date"].max()

print(f"The xml dataset has coverage from {min_time_xml.date()} to {max_time_xml.date()}")

The xml dataset has coverage from 2009-10-06 to 2026-02-26


# Merging datasets

In [20]:
#rename column(s) in each to match the counterpart

print(df_xml.columns)
print(df_corp.columns) # party.facts.id to partyfacts_ID

Index(['paragraph_nr', 'date', 'speaker', 'party', 'role', 'text',
       'source_file', 'chair'],
      dtype='str')
Index(['date', 'agenda', 'speechnumber', 'speaker', 'party', 'chair', 'text'], dtype='str')


In [ ]:
# Dropping irrelevant columnds before merge

df_corp = df_corp.drop(columns=["party.facts.id", "parliament", "iso3country", "terms"])
print(df_corp.columns) #double check

In [22]:

# Keep only rows in df2 strictly after max date of df1
df_xml_filtered = df_xml[df_xml['date'] > max_time_corp]

# Concatenate
df_merged = pd.concat([df_corp, df_xml_filtered], ignore_index=True)

# sort by date
df_merged = df_merged.sort_values('date').reset_index(drop=True)

In [23]:
#check that the span now is the entire period
#This dataset spans the following time period

min_time_merged = df_merged["date"].min()
max_time_merged = df_merged["date"].max()

print(f"The merged datasets has coverage from {min_time_merged.date()} to {max_time_merged.date()}")

The merged datasets has coverage from 1997-10-07 to 2026-02-26


### As the future analysis will discard any utterences from the chair of parliament, as these serve to aid order and time of speaking, chair == True will be discarded now for efficiency reasons. Additionally, they typically speak for a very long time at the beginning fo each recorded session

In [25]:

df_merged = df_merged[df_merged['chair'] == False] # Only include chair == False

'''

df_merged = df_merged.loc[
    (df_merged["chair"].isna()) | ((df_merged["chair"] == "False"))]

df_merged = df_merged[df_merged['role'] != "formand"] # Same idea, formatted differently in the xml dataset
'''


'\n\ndf_merged = df_merged.loc[\n    (df_merged["chair"].isna()) | ((df_merged["chair"] == "False"))]\n\ndf_merged = df_merged[df_merged[\'role\'] != "formand"] # Same idea, formatted differently in the xml dataset\n'

In [26]:
len(df_merged)

639400

In [27]:
df_merged

,date,agenda,speechnumber,speaker,party,chair,text,paragraph_nr,role,source_file
2,1997-10-07,Statsministerens redegørelse i henhold til gru...,3.0,Poul Nyrup Rasmussen,S,False,For 25 år siden sagde et flertal i befolkninge...,NaN,NaN,NaN
3,1997-10-09,2) Forhandling om redegørelse nr. R 1.,136.0,Frank Dahlgaard,KF,False,»De Konservative kræver den sorte skole tilbag...,NaN,NaN,NaN
4,1997-10-09,2) Forhandling om redegørelse nr. R 1.,137.0,Kristian Thulesen Dahl,DF,False,Jeg må give hr. Holger K. Nielsen fuldstændig ...,NaN,NaN,NaN
5,1997-10-09,2) Forhandling om redegørelse nr. R 1.,138.0,Holger K. Nielsen,SF,False,"Til hr. Frank Dahlgaard kan jeg sige, at når m...",NaN,NaN,NaN
6,1997-10-09,2) Forhandling om redegørelse nr. R 1.,139.0,Frank Dahlgaard,KF,False,"Det er jo forkert, hvad hr. Holger K. Nielsen ...",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
1146394,2026-02-26,NaN,NaN,Rasmus Jarlov,KF,False,"Mener Socialdemokratiet, at man gør noget godt...",76.0,medlem,20251_M57_helemoedet.xml
1146396,2026-02-26,NaN,NaN,Benny Engelbrecht,S,False,"Mener hr. Rasmus Jarlov, at Konservative, den ...",78.0,medlem,20251_M57_helemoedet.xml
1146398,2026-02-26,NaN,NaN,Rasmus Jarlov,KF,False,"Jeg synes ikke, det burde være så svært at for...",80.0,medlem,20251_M57_helemoedet.xml
1146400,2026-02-26,NaN,NaN,Benny Engelbrecht,S,False,"Vi kan ikke vide, præcis hvor omkostningsfuldt...",82.0,medlem,20251_M57_helemoedet.xml


In [29]:
#Save the data

merged_data_path = os.path.join("..",
                            "..",
                            "..",
                            "..",
                            "data",
                            "csv_meetings"
                            "merged_speech_data.csv")
                                
df_merged.to_csv(merged_data_path, index=False)

# Now for sentence segmentation

Implementatino of the DaCy pipeline. Due to incompatibility with newer python versions, python version 3.10.19 used. Spacy version 3.5.4 has been used, as the model card on huggingface for da_dacy_large_trf specifies spacy to be >=3.5.2,<3.6.0.


*Personal note: use env: dacy_env*


In [ ]:
for model in dacy.models():
    print(model)

## Issues

While this downloading process should be straight forward with nlp = spacy.load("da_dacy_large_trf-0.2.0"), this does not work.
Thus, the da_dacy_large_trf-any-py3-none-any.whl file has been manually from huggingface, and the path specified
- dacy_path = os.path.abspath("C:/Users/marku/Desktop/AU/Bachelor/Data_outside_git/da_dacy_large_trf-any-py3-none-any.whl")

The file must be renamed for the pipeline to accept it
- #rename da_dacy_large_trf-any-py3-none-any.whl da_dacy_large_trf-0.2.0-py3-none-any.whl

And then manually installed
- python -m pip install da_dacy_large_trf-0.2.0-py3-none-any.whl --no-deps

Verification that spacy sees the model
- import spacy
- print(spacy.util.get_installed_models())

However, dacy still does not recognize, and since it has been build on spacy infrastructure, spacy is used for loading the model instead
- nlp = spacy.load("da_dacy_large_trf")

The name of the model can be double chekced
- print(nlp.meta["name"]) # shoudl say the above name

In [ ]:
import spacy

In [ ]:
nlp = spacy.load("da_dacy_large_trf", disable=["ner", "trainable_lemmatizer", "morphologizer", "coref", "entity_linker"]) #dacy load does not work, but spacy with the dacy corpus
print(nlp.meta["name"]) # shoudl say the above name

In [ ]:
#Setup example
doc = nlp(
    "Jeg skal blot beklage, at hr. Poul Erik Dyrlund tilsyneladende ikke forstår sit eget lovforslag. Noget efterfølgende sætninger."
    )

In [ ]:
for sent in doc.sents:
    print(sent)

# Be very careful about running pip installs as the environment is very unstable, with many legacy dependencies which can override eachother

In order to use GPU for sentence segmentation, check the following
- import torch
- print(torch.cuda.is_available())
- print(torch.cuda.get_device_name(0))

Check your CUDA version
- nvidia-smi

Install matching spaCy GPU package, e.g. for CUDA 11.8:
- pip install spacy[cuda118]

or CUDA 12.x:
- pip install spacy[cuda12x]


Then initialize dacy model using spacy
- import spacy
- import dacy

- spacy.prefer_gpu()  # must be called BEFORE loading the model
- nlp = spacy.load("da_dacy_large_trf", disable=["ner", "trainable_lemmatizer", "morphologizer", "coref", "entity_linker"]) 
- print(nlp.meta["name"]) 

Confirm use of GPU
- import cupy
- print(cupy.cuda.runtime.getDeviceCount())  # should be > 0

I got a pkg-packages building wheel problem when building spacy through regular pip install on cuda version (pip install spacy[cuda13x] / pip install spacy[cuda-autodetect])
- This was fixed by going around the building wheel by doing the following:
    - For CUDA 13.0:
        - pip install cupy-cuda13x
        - Then pip install spacy==3.5.4

To check what torch version got installed (CPU only or GPU enabled)
-   import torch
    print(torch.__version__) # This line will print something along the lines of x.xx.x+cpu if CPU build is installed
    print(torch.version.cuda) # This will print None if GPU installation is not found

If torch.version.cuda is None:
- Run pip uninstall torch torchvision torchaudio -y in your environment
- Check Cuda version (In command prompt) with nvidia-smi
- Go to https://pytorch.org/get-started/locally/ To find the correct version and run the pip command in terminal from that website

In [ ]:
############# RUN FIRST OR KERNELS DIE ##############

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
import torch
import cupy
import spacy
from pathlib import Path

print(torch.cuda.is_available()) # Base CUDA CHECK
print(cupy.cuda.runtime.getDeviceCount()) # CUPY CUDA CHECK

spacy.require_gpu() # REUQIRE GPU GOING FORWARD
print("spaCy GPU enabled")

True
1
spaCy GPU enabled


In [3]:
'''import importlib
import sentence_segmentation

importlib.reload(sentence_segmentation)'''

from sentence_segmentation import SentenceSegmentation


merged_data_path = os.path.join("..",
                            "..",
                            "..",
                            "..",
                            "Data_outside_git",
                            "merged_speech_data.csv")

output_path = Path(os.path.join("..",
                           "..",
                           "raw_data",
                           "test_sentences.jsonl"))

SS = SentenceSegmentation(input_path=merged_data_path, output_path=output_path)

#Confirm use of GPU
#import cupy
#print(cupy.cuda.runtime.getDeviceCount())  # should be > 0

SS.sentence_segmentation(n_rows = 2, batch_size=2, n_process=1)

c:\Users\runet\anaconda3\envs\spacy_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


attempting to initialize spacy with gpu... 
True
loading 'da_dacy_large_trf'...
The loaded model name is: dacy_large_trf
reading csv file...



c:\dev\bachelor\Bachelor_project\data_making\scripts\initial_data_creation\sentence_segmentation.py:40: DtypeWarning: Columns (9,10,11,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  self.df = pd.read_csv(self.input_path)


extracting texts from csv file... 

Starting segmentation...

initializing pipeline...



  0%|          | 0/2 [00:00<?, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (585 > 512). Running this sequence through the model will result in indexing errors
c:\Users\runet\anaconda3\envs\spacy_gpu\lib\site-packages\thinc\shims\pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):
c:\Users\runet\anaconda3\envs\spacy_gpu\lib\site-packages\thinc\util.py:398: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toDlpack()  # type: ignore
100%|██████████| 2/2 [00:05<00:00,  2.88s/it]


#### OBS can also be run from terminal with the following command

python sentence_segmentation.py --inpath_to_csv <path/to/merged_data> --outpath_to_jsonl <path/for/saving/jsonl> 

Optional parameters:

--n_rows number-of-paragraphs-to-run (deafault -1)

--batch_size batch-size-for-pipe (default 2)

--n_process multiprocessing (default 1)







# Data cleaning

#### OBS former cleaning included discarding chair == True
#### The sentences matching the following characteristics will be discarded

- Less than three characters (need some guidelines here. Maybe just empty strings)
- Contains parathesis
- Remove leading/trailing whitespaces along with cases of multiple whitespaces in text




In [ ]:
#set path
from pathlib import Path
import os
import json
import re

json_data_path = os.path.join("..",
                              "..",
                              "raw_data",
                              "test_sentences.jsonl")

json_output_path = os.path.join("..",
                              "..",
                              "raw_data",
                              "filtered_test_sentences.jsonl")

In [ ]:
char_limit = 5

input_path = Path(json_data_path)
output_path = Path(json_output_path)

with input_path.open("r", encoding="utf-8") as f_in, \
     output_path.open("w", encoding="utf-8") as f_out:
    
    for line in f_in:
        entry = json.loads(line)
        text = entry.get("text", "")
        
        if (len(text.strip()) > char_limit) and ("(" not in text) and (")" not in text):
            #remove multiple following whitespaces
            text = re.sub(r'\s+', ' ', text).strip()
            entry["text"] = text
            f_out.write(json.dumps(entry, ensure_ascii=False) + "\n")
        
        else:
            pass

# Test-Train-Split


#### However, validation set will be drawn from the training data due to the need for for prelimenary blame detection allowing computationally assisted lableing and artificially inflated probability of blame in validationset


In [ ]:
import json
import random
from pathlib import Path
import tqdm

#setting paths
input_path = Path(json_output_path)

train_path = Path(os.path.join(json_output_path,
                               "..",
                               "test_json_modelling_data.jsonl"))

inference_path = Path(os.path.join(json_output_path,
                               "..",
                               "test_json_inference_data.jsonl"))

train_sample_size = 50 #OBS set as desired size of train/test/validation set
random.seed(42)

# First pass: count total lines
with input_path.open("r", encoding="utf-8") as f:
    total_lines = sum(1 for _ in f)

# Randomly select indices for train/val/test
train_indices = set(random.sample(range(total_lines), train_sample_size))

# Second pass: write to respective files
with input_path.open("r", encoding="utf-8") as f_in, \
     train_path.open("w", encoding="utf-8") as f_train, \
     inference_path.open("w", encoding="utf-8") as f_inf:

    for i, line in enumerate(tqdm(f_in, total=total_lines)):
        if i in train_indices:
            f_train.write(line)
        else:
            f_inf.write(line)

print(f"Train/val/test: {train_sample_size} lines")
print(f"Inference: {total_lines - train_sample_size} lines")